# Kafka and PySpark Streaming Example

This notebook demonstrates how to read messages from a Kafka topic, display the messages, and parse JSON data using PySpark. 

## Initialization

First, we initialize a Spark session with the necessary Kafka package.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StringType
import time

spark = SparkSession.builder \
    .appName("KafkaSparkStreamingExample") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0")\
    .getOrCreate()

## Event Data, JSON Data Structure and Schema

We are dealing with a store submitting order updates. 
This store provides details such as order time, order ID, item ID, order units, and address information (city, state, and zipcode).


The Event, JSON data we are dealing with has the following structure:

```json
{
  "ordertime": 1516952774174,
  "orderid": 41,
  "itemid": "Item_7",
  "orderunits": 7.1615426137476,
  "address": {
    "city": "City_7",
    "state": "State_",
    "zipcode": 39126
  }
}


This JSON represents an order with the following fields:

* ordertime: The timestamp of the order.
* orderid: The ID of the order.
* itemid: The ID of the item.
* orderunits: The number of units ordered.
* address: A **nested JSON object** containing the address details (city, state, zipcode).


#### Streaming and loading the data into a dataframe

We have set up a streaming DataFrame `df` to continuously read data from the specified Kafka topic.
Here’s how we do it:

You're configuring the Kafka parameters and creating a Spark streaming DataFrame that subscribes to a Kafka topic using PySpark. Here’s a breakdown of the steps and components involved:

For now, we assume that the topic is a continuous stream of JSON-formatted messages representing commerce transactions, where each message contains details about individual orders.


### Kafka Parameters Setup

- **`kafka.bootstrap.servers`**: This specifies the Kafka server's address, including the port. It's used by Spark to locate the Kafka cluster.
- **`subscribe`**: This parameter indicates the name of the Kafka topic (`commerce-fhtw`) that Spark should subscribe to. It is crucial for directing Spark to the right stream of data.
- **`kafka.security.protocol`**: Set to `SASL_SSL` to indicate that the connection to Kafka should be secured using SASL (Simple Authentication and Security Layer) over SSL (Secure Sockets Layer).
- **`kafka.sasl.mechanism`**: This specifies the SASL mechanism to use for authentication; in this case, `PLAIN`, which means it will use a simple username and password method.
- **`kafka.sasl.jaas.config`**: Provides the

In [ ]:
kafka_params = {
      "kafka.bootstrap.servers": "46.225.20.89:9092",
      "subscribe": "commerce-fhtw",
      "kafka.security.protocol": "PLAINTEXT",
      "startingOffsets": "latest",
}

In [ ]:
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# How does the data look like?

We initiate the process by displaying all incoming messages to gain a preliminary understanding of the data flow. This step provides a basic overview of the ongoing activity within the Kafka topic.

In [ ]:
query = df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()


This output contains several important pieces of information for each message:

- **`key`**: The key of the message in the Kafka topic, represented in a byte array format.
- **`value`**: The actual content of the message, also in a byte array format. This is where our JSON data is stored.
- **`topic`**: The name of the Kafka topic from which the message was read.
- **`partition`**: The partition number in the Kafka topic where the message was stored.
- **`offset`**: The offset of the message within the partition, which serves as a unique identifier for the message within that partition.
- **`timestamp`**: The timestamp indicating when the message was produced.
- **`timestampType`**: The type of timestamp (e.g., whether it is the creation time or log append time).

By examining this output, you can get a basic idea of the structure of the messages coming from the Kafka topic, including the metadata and the actual data content.


As you will see the messages are keep coming in and coming in  and we want to terminate that in order to allow for another DF to process.

In [ ]:
query.stop()

Now, in order to make our lives easier, we will write a function that allows us to display all the incoming data instead of having to write this whole code section every now and then.


In [ ]:
def display_stream(df, timeout=10):
    query = df.writeStream \
        .outputMode("append") \
        .format("console") \
        .option("truncate", "false") \
        .start()
    # Let the stream run for a short period and then stop (for demonstration purposes)
    time.sleep(timeout)
    query.stop()

### Read the json output and event data

Now, we actually want to analyze the section that contains the value, which holds the actual Kafka message and event data. As you can see, the message content is in the `value` entry. Therefore, we need to read and process the data from the `value` entry.

Here’s how we can do this:

In [ ]:
# Select and cast the key and value columns to STRING
df_message = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

display_stream(df_message, 10)

## Extract the actual data from the json

Alright, now we get the actual JSON message, but we want to deal with it like a data frame. So we would actually like to extract the data inside the JSON data as a column.
as a DF.


### The Schema
We define the schema for the incoming JSON data. This schema matches the structure of the JSON messages we expect from Kafka.

The schema is defined to match the structure of the incoming JSON data. It specifies the data types for each field in the JSON object.
This is necessary because it allows Spark to correctly parse and interpret the data, ensuring that each field is handled appropriately (e.g., as an integer, string, etc.).
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType
# Define the schema for the JSON data
schema = StructType([
    StructField("ordertime", LongType()),
    StructField("orderid", IntegerType()),
    StructField("itemid", StringType()),
    StructField("orderunits", DoubleType()),
    StructField("address", StructType([
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("zipcode", IntegerType())
    ]))
])

In [ ]:
df_value = df.selectExpr("CAST(value AS STRING) as json_string")
# Parse the JSON data and apply schema
parsed_df = df_value.withColumn("jsonData", from_json(col("json_string"), schema)).select("jsonData.*")
display_stream(parsed_df, 10)

## Analyzing the data

So since just streaming and displaying the data is quite boring, we want to actually do some stuff with it. Let's try to deal with it.


In [ ]:
from pyspark.sql.functions import sum

# Aggregate the order units
order_units_df = parsed_df.groupBy().agg(sum("orderunits").alias("total_order_units"))

# Display the aggregated data
# complete mode is required for aggregations!!
query = order_units_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

time.sleep(20)
query.stop()

### Your Turn

Try to calculate the orders per city!

In [ ]:
# Your task is to calculate the total order units per city

Calculating Average Order Units